In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
import gc
gc.collect()


In [ ]:
import gc # gc.collect()
from intecomm_analytics.constants import HTN_ALONE, DM_ALONE, HTN_DM
from edc_appointment.constants import CANCELLED_APPT, SKIPPED_APPT
from intecomm_analytics.constants import HIV_ALONE
from intecomm_analytics.dataframes import (
    get_all_unscheduled,
    get_all_scheduled,
    get_appt_df,
    get_scheduled_and_missed,
    get_subjects_who_missed_scheduled,
    get_referrals,
    get_drug_refills_unscheduled,
    get_community_who_visited_facility,
)

In [ ]:
df_appt = get_appt_df()

In [ ]:
# verify cohort numbers still tally
# and show the total number of appointments for HIV
df_tmp = (df_appt
 .groupby(by=["subject_identifier", "primary_cohort_str"])
 .size()
 .to_frame()
 .sort_values(by=["subject_identifier", "primary_cohort_str"])
 .reset_index()
 .rename(columns={0: "freq"})
 .query("primary_cohort_str.isin(['HIV_ALONE'])")
 .groupby("subject_identifier").sum()
 )
assert len(df_tmp) == 489 # nosec B101
print(f"{df_tmp.freq.sum()} total appointments (HIV ALONE)")

In [ ]:
# verify cohort numbers still tally
# and show the total number of appointments for NCD
df_tmp = (df_appt
 .groupby(by=["subject_identifier", "primary_cohort_str"])
 .size()
 .to_frame()
 .sort_values(by=["subject_identifier", "primary_cohort_str"])
 .reset_index()
 .rename(columns={0: "freq"})
 .query("primary_cohort_str.isin(['HTN_ALONE','DM_ALONE','HTN_DM'])")
 .groupby("subject_identifier").sum()
 )
assert len(df_tmp) == 1211 # NCD # nosec B101
print(f"{df_tmp.freq.sum()} total appointments (NCD)")

In [ ]:
# the total number of scheduled/unscheduled visits is:

# confirm can use visit_code_sequence or appt_reason and get same tally
assert len(df_appt[
    (df_appt.visit_code_sequence==0.0) &
    ~(df_appt.appt_status.isin([CANCELLED_APPT, SKIPPED_APPT]))
    ]
    .reset_index(drop=True)
) == 18998 # noqa

assert len(df_appt[
    (df_appt.appt_reason=="scheduled") &
    ~(df_appt.appt_status.isin([CANCELLED_APPT, SKIPPED_APPT]))
    ]
    .reset_index(drop=True)
) == 18998 # nosec B101

# the total number of scheduled visits is:
assert len(df_appt[
    (df_appt.visit_code_sequence>0.0) &
    ~(df_appt.appt_status.isin([CANCELLED_APPT, SKIPPED_APPT]))
    ]
    .reset_index(drop=True)
) == 268 # nosec B101

assert len(df_appt[
    (df_appt.appt_reason=="unscheduled") &
    ~(df_appt.appt_status.isin([CANCELLED_APPT, SKIPPED_APPT]))
    ]
    .reset_index(drop=True)
) == 268 # nosec B101

# appts (all cohorts) by appt_reason

df_appt.groupby("appt_reason").size()

In [ ]:
df_hiv = (df_appt[
    (df_appt.primary_cohort.isin([HIV_ALONE]))]
    .copy()
    .reset_index(drop=True)
)
print(f"{len(df_hiv)} scheduled/unscheduled appointments (HIV)")


In [ ]:
df_htn_dm = (df_appt[(df_appt.primary_cohort.isin([HTN_ALONE, DM_ALONE, HTN_DM]))]
    .copy()
    .reset_index(drop=True)
)
print(f"{len(df_htn_dm)} scheduled/unscheduled appointments (NCD)")


In [ ]:
# 1
df_tmp = get_all_scheduled(df_htn_dm)
df_tmp

In [ ]:
# 2
df_tmp = get_scheduled_and_missed(df_htn_dm)
df_tmp

In [ ]:
# 3. Number of participants who missed one or more appointments
df_tmp = get_subjects_who_missed_scheduled(df_htn_dm)
df_tmp

In [ ]:
# 4. Number of participants who visited the facility unscheduled one or more times for any reason
df_tmp = get_all_unscheduled(df_htn_dm)
df_tmp

In [ ]:
# 5. Number of participants who visited the facility unscheduled by referral or self-referral
df_tmp = get_referrals(df_htn_dm)
df_tmp

In [ ]:
# 6. Number of community participants who visited the facility to pick up medicines
df_tmp = get_drug_refills_unscheduled(df_htn_dm)
df_tmp

In [ ]:
# 7. Number of community participants who visited the facility for any other reason
df_tmp = get_community_who_visited_facility(df_htn_dm)
df_tmp


In [ ]:
# HIV_ALONE .....

In [ ]:
# 1
df_tmp = get_all_scheduled(df_hiv)
df_tmp

In [ ]:
# 2
df_tmp = get_scheduled_and_missed(df_hiv)
df_tmp

In [ ]:
# 3. Number of participants who missed one or more appointments
df_tmp = get_subjects_who_missed_scheduled(df_hiv)
df_tmp

In [ ]:
# 4. Number of participants who visited the facility unscheduled one or more times for any reason
df_tmp = get_all_unscheduled(df_hiv)
df_tmp

In [ ]:
# 5. Number of participants who visited the facility unscheduled by referral or self-referral
df_tmp = get_referrals(df_hiv)
df_tmp

In [ ]:
# 6. Number of community participants who visited the facility to pick up medicines
df_tmp = get_drug_refills_unscheduled(df_hiv)
df_tmp

In [ ]:
# 7. Number of community participants who visited the facility for any other reason
df_tmp = get_community_who_visited_facility(df_hiv)
df_tmp


In [ ]:
# appts missed by appt_reason, assignment
df_tmp = df_htn_dm.query("appt_timing=='missed'").groupby(by=["assignment", "appt_reason"]).size().to_frame().reset_index().rename(columns={0: "visit_count"})
df_tmp1= df_htn_dm.query("appt_timing=='missed'").groupby(by=["assignment", "appt_reason"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier": "subjects"})
df_tmp = df_tmp.merge(df_tmp1, on=["assignment", "appt_reason"], how="left")
df_tmp = df_tmp.merge(df_htn_dm.groupby(by=["assignment"]).subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"total_subjects"}), on=["assignment"], how="left")
df_tmp["total_visits"] = df_tmp.groupby("assignment")["visit_count"].transform("sum")
df_tmp["prop_of_subjects"] = df_tmp.subjects/df_tmp.total_subjects
df_tmp

In [ ]:
df_htn_dm.query("location_direction=='a->b'").location_comment.value_counts().to_frame()["count"].sum()

In [ ]:
# NCD, prop of appts scheduled / unscheduled
df_tmp = df_htn_dm.groupby("appt_reason").size().to_frame().reset_index().rename(columns={0:"freq"})
df_tmp["total"] = len(df_htn_dm)
df_tmp["prop"] = df_tmp.freq/df_tmp.total
df_tmp

In [ ]:
# NCD, prop of appts scheduled / unscheduled by assignment
df_tmp = df_htn_dm.groupby(by=["appt_reason", "assignment"]).size().to_frame().reset_index().rename(columns={0:"freq"})
df_tmp["total"] = len(df_htn_dm)
df_tmp["prop"] = df_tmp.freq/df_tmp.total
df_tmp

In [ ]:
df_tmp = df_htn_dm.query("appt_type=='clinic' and timepoint>=0.0 and timepoint<12.0").groupby(by=["appt_reason", "assignment"]).size().to_frame().reset_index().rename(columns={0:"freq"})
df_tmp["total"] = len(df_htn_dm)
df_tmp["prop"] = df_tmp.freq/df_tmp.total
df_tmp

In [ ]:
df_tmp = df_htn_dm.query("appt_type=='clinic' and timepoint>=0.0 and timepoint<12.0").groupby(by=["appt_reason", "assignment"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier":"freq"})
df_tmp["total"] = len(df_htn_dm)
df_tmp["prop"] = df_tmp.freq/df_tmp.total
df_tmp

In [ ]:
# Number of participants who visited the facility unscheduled for any reason ///// one or more times (i.e. self-referred when unwell)
df_tmp = df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and location_direction=='a->b'").groupby(by=["assignment"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier":"subjects"})
df_tmp["total_visits"] = len(df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and location_direction=='a->b'"))

df_tmp1 = df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and assignment=='b' and appt_reason=='unscheduled'").groupby(by=["assignment"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier":"subjects"})
df_tmp1["total_visits"] = len(df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and assignment=='b' and appt_reason=='unscheduled'"))

df_tmp = pd.concat([df_tmp, df_tmp1])
df_tmp = df_tmp.merge(df_htn_dm.groupby(by=["assignment"]).subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"total_subjects"}), on=["assignment"], how="left")
df_tmp["prop_subjects"] = df_tmp.subjects/df_tmp.total_subjects
df_tmp


In [ ]:
# Number of participants who visited the facility unscheduled unwell as a referral or self-referral
df_tmp = pd.concat([
    df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and location_direction=='a->b' and assignment=='a'"),
    df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and assignment=='b' and appt_reason=='unscheduled'")])
df_tmp

In [ ]:
# Number of participants who visited the facility unscheduled by referral or self-referral
df_tmp = df_htn_dm.query("reason_unscheduled=='patient_unwell_outpatient'").groupby("assignment").subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"subjects"})
df_tmp1 = df_htn_dm.query("reason_unscheduled=='patient_unwell_outpatient'").groupby("assignment").size().to_frame().rename(columns={0:"total_visits"})
df_tmp = df_tmp.merge(df_tmp1, on=["assignment"], how="left", suffixes=["", "_y"])
df_tmp = df_tmp.merge(df_htn_dm.groupby(by=["assignment"]).subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"total_subjects"}), on=["assignment"], how="left")
df_tmp["prop"] = df_tmp.subjects/df_tmp.total_subjects
df_tmp

In [ ]:
# Number of participants who visited the facility unscheduled by referral or self-referral
df_tmp = df_htn_dm.query("reason_unscheduled=='patient_unwell_outpatient'").groupby("assignment").subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"subjects"})
df_tmp1 = df_htn_dm.query("reason_unscheduled=='patient_unwell_outpatient'").groupby("assignment").size().to_frame().rename(columns={0:"total_visits"})
df_tmp = df_tmp.merge(df_tmp1, on=["assignment"], how="left", suffixes=["", "_y"])
df_tmp = df_tmp.merge(df_htn_dm.groupby(by=["assignment"]).subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"total_subjects"}), on=["assignment"], how="left")
df_tmp["prop"] = df_tmp.subjects/df_tmp.total_subjects
df_tmp

In [ ]:
df_tmp1 = pd.concat([df_tmp.query("reason_unscheduled=='patient_unwell_outpatient'"), df_tmp.query("location_comment.isin(['self_referral', 'referral'])")])


In [ ]:
df_tmp[df_tmp["location_comment"].str.contains('referral', na=False)].groupby("assignment").reason_missed_detail.value_counts(dropna=False)


In [ ]:
df_tmp.reason_missed_detail.value_counts(dropna=False)

In [ ]:
df_htn_dm.query("timepoint>=0.0 and timepoint<12.0 and location_direction=='a->b'")

In [ ]:
# NCD, freq of subjects with appts scheduled / unscheduled
df_tmp = df_htn_dm.groupby(by=["subject_identifier", "appt_reason"]).subject_identifier.count().groupby(by=["appt_reason"]).size()
df_tmp

In [ ]:
# NCD, freq of subjects with appts scheduled / unscheduled by assignment
df_tmp = df_htn_dm.groupby(by=["subject_identifier", "appt_reason", "assignment"]).subject_identifier.count().groupby(by=["appt_reason",  "assignment"]).size()
df_tmp


In [ ]:
# NCD, prop of appts scheduled by assignment
df_tmp = df_htn_dm.query("appt_reason=='scheduled' and appt_timing=='missed'").groupby(by=["assignment"]).size().to_frame().reset_index().rename(columns={0:"freq"})
df_tmp1 = df_htn_dm.query("appt_reason=='scheduled'").groupby(by=["assignment"]).size().to_frame().reset_index().rename(columns={0:"freq"})
df_tmp = df_tmp.merge(df_tmp1, on=["assignment"], how="left", suffixes=["", "_y"]).rename(columns={"freq_y":"total"})
df_tmp["prop"] = df_tmp.freq/df_tmp.total
df_tmp

In [ ]:
df_tmp = df_htn_dm.query("appt_reason=='scheduled' and appt_timing=='missed'").groupby(by=["assignment"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier":"freq"})
df_tmp1 = df_htn_dm.query("appt_reason=='scheduled'").groupby(by=["assignment"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier":"freq"})
df_tmp = df_tmp.merge(df_tmp1, on=["assignment"], how="left", suffixes=["", "_y"]).rename(columns={"freq_y":"total"})
df_tmp["prop"] = df_tmp.freq/df_tmp.total
df_tmp

In [ ]:
df_tmp = df_htn_dm.query("appt_reason=='unscheduled'").groupby(by=["assignment"]).subject_identifier.nunique().to_frame().reset_index().rename(columns={"subject_identifier":"freq"})
df_tmp1 = df_htn_dm.query("appt_reason=='unscheduled'").groupby(by=["assignment"])[["subject_identifier"]].size().reset_index().rename(columns={0:"freq"})
df_tmp = df_tmp.merge(df_tmp1, on=["assignment"], how="left", suffixes=["", "_y"]).rename(columns={"freq_y":"total_visits"})
df_tmp["prop_total_visits"] = df_tmp.freq/df_tmp.total_visits
df_tmp = df_tmp.merge(df_htn_dm.groupby(by=["assignment"]).subject_identifier.nunique().to_frame().rename(columns={"subject_identifier":"total_subjects"}), on=["assignment"], how="left")
df_tmp["prop_subjects"] = df_tmp.freq/df_tmp.total_subjects
df_tmp


In [ ]:
df_htn_dm

In [ ]:
# Number of scheduled appointments
df_summary = (df_scheduled
    .groupby("assignment")
    .size()
    .to_frame()
    .pivot_table(columns="assignment", values=0)
)
df_summary.columns.name=""
df_summary["label"] = "Number of scheduled appointments"
df_summary

In [ ]:
# Number of scheduled appointments missed
df_tmp = (
    df_scheduled[(df_scheduled.appt_timing=="missed")]
    .groupby("assignment")
    .appt_timing
    .value_counts()
    .to_frame()
    .pivot_table(columns="assignment", values="count")
)
df_tmp.columns.name=""
df_tmp["label"] = "Number of scheduled appointments missed"
df_summary = pd.concat([df_summary, df_tmp])
df_summary

In [ ]:
# Number of participants who missed one or more appointments
df_tmp = (
    df_scheduled[(df_scheduled.appt_timing=="missed")]
    .groupby("assignment")
    .subject_identifier.nunique()
    .to_frame()
    .pivot_table(columns="assignment", values="subject_identifier")
)
df_tmp.columns.name=""
df_tmp["label"] = "Number of participants who missed one or more appointments"
df_summary = pd.concat([df_summary, df_tmp])
df_summary[["label", "a", "b"]]


In [ ]:
df_scheduled[(df_scheduled.appt_timing=="missed")][["visit_code", "assignment", "appt_timing", "reason_missed", "reason_missed_other", "reason_missed_detail", "visit_datetime"]]

In [ ]:
# Number of participants who visited the facility unscheduled one or more times (i.e. self-referred when unwell) HTN_ALONE,DM_ALONE, HTN_DM
df_tmp = (
    df_unscheduled[(df_unscheduled.appt_timing!="missed")]
    .groupby("assignment")
    .subject_identifier.nunique()
    .to_frame()
    .pivot_table(columns="assignment", values="subject_identifier")
)
df_tmp.columns.name=""
df_tmp["label"] = "Number of participants who visited the facility unscheduled one or more times (i.e. self-referred when unwell)"
df_summary = pd.concat([df_summary, df_tmp])
df_summary[["label", "a", "b"]]


In [ ]:
# Number of participants from the community arm who attended the facility on one or more occasions for other reasons HTN_ALONE,DM_ALONE, HTN_DM
df_tmp = (
    df_htn_dm[(df_htn_dm.location_direction=="a->b") & (df_htn_dm.appt_timing=="ontime")]
    .groupby("assignment")
    .subject_identifier.nunique()
    .to_frame()
    .pivot_table(columns="assignment", values="subject_identifier")
)
df_tmp.columns.name=""
df_tmp["label"] = "Number of participants from the community arm who attended the facility on one or more occasions for other reasons"
df_summary = pd.concat([df_summary, df_tmp])
df_summary[["label", "a", "b"]]


In [ ]:
total = df_htn_dm[(df_htn_dm.appt_timing=="ontime")].groupby(by=["assignment", "location_direction"]).subject_identifier.count()
unique_subjects = df_htn_dm[(df_htn_dm.appt_timing=="ontime")].groupby(by=["assignment", "location_direction"]).subject_identifier.nunique()
print(unique_subjects, total)

In [ ]:
df_summary

In [ ]:
from great_tables import GT, html, loc, style

GT(df_summary[["label", "a", "b"]])

In [ ]:
from intecomm_analytics.notebooks.primary.table_utils import \
    get_primary_cohorts_cells_for_continuous_var

get_primary_cohorts_cells_for_continuous_var()